<a href="https://colab.research.google.com/github/Shizukesasss/Lab-Activity---Your-Data-Our-Data-Part-1-and-Part2/blob/Lab-Activity-3---Your-Data-Our-Data/ACTIVITY_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pyspark.sql import SparkSession

# Start the Spark Session
spark = SparkSession.builder.appName("HospitalAnalysis").getOrCreate()

# Load the dataset
# header=True uses the first row as column names
# inferSchema=True detects numbers vs strings automatically
rdd = spark.read.csv("hospital_readmission_dataset.csv", header=True, inferSchema=True)

In [ ]:
# 2. Partition by Range based on Age
# This sorts the data and splits it into 5 logical age-based buckets.
rddRangeAge = rdd.repartitionByRange(5, "age")

In [ ]:
# Transformation: Filtering for seniors
# Because the data is range-partitioned, Spark knows exactly which partitions to scan.
SeniorsOnly = rddRangeAge.filter("age >= 65")

In [ ]:
# Transformation: Sort within partitions
FinalSeniors = SeniorsOnly.sort("age")

# Result
FinalSeniors.show()

+----------+--------------+------+---+------+-------+-----------------+-------------------+--------------+--------------+-----------------+-------------------------+-----------------+--------------+---------------------+----------------------+-----+
|patient_id|admission_date|season|age|gender| region|primary_diagnosis|comorbidities_count|length_of_stay|treatment_type|medications_count|followup_visits_last_year|prev_readmissions|insurance_type|discharge_disposition|readmission_risk_score|label|
+----------+--------------+------+---+------+-------+-----------------+-------------------+--------------+--------------+-----------------+-------------------------+-----------------+--------------+---------------------+----------------------+-----+
|    P00788|    2021-07-19|Summer| 65|  Male|   East|   Kidney Disease|                  5|             9|       Medical|                8|                        4|                1|      Medicaid|          Home Health|                  0.97|    1|


In [ ]:
ReducedPartition = FinalSeniors.coalesce(2)

In [ ]:
UniqueSeniors = ReducedPartition.distinct()

In [ ]:
TotUniSen = UniqueSeniors.count()

print("Total unique senior records:", TotUniSen)

Total unique senior records: 2712


In [ ]:
#ACTIVITY #4 BIG DATA ANALYSIS
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, min, max
import shutil
from google.colab import files

spark = (SparkSession.builder.appName("Activity #4 ni Nash Mapula").getOrCreate())

schema = "CustomerID INT, Gender STRING, Age DOUBLE, Annual_Income DOUBLE, Spending_Score Double"

DF = spark.read.csv("store_customers.csv", header=True, schema=schema)


DF_distinct = DF.distinct()


DFCleaned = DF_distinct.dropna()

print("Cleaned Data Sample")
DFCleaned.show(5)


Cleaned Data Sample
+----------+------+----+-------------+--------------+
|CustomerID|Gender| Age|Annual_Income|Spending_Score|
+----------+------+----+-------------+--------------+
|      1341|     M|37.0|         33.5|          56.0|
|      1378|     M|53.0|         86.1|          28.0|
|      1541|     M|26.0|         22.1|          73.0|
|      1576|     M|37.0|         44.9|          37.0|
|      1171|     M|28.0|         26.4|          46.0|
+----------+------+----+-------------+--------------+
only showing top 5 rows


In [ ]:
print("Insight 1: Employee Count by CustomerID ")
INSIGHT1 = DFCleaned.groupBy("CustomerID").count().orderBy("count", ascending=False)
INSIGHT1.show()

Insight 1: Employee Count by CustomerID 
+----------+-----+
|CustomerID|count|
+----------+-----+
|      1959|    1|
|      1591|    1|
|      1829|    1|
|      1580|    1|
|      1342|    1|
|      1238|    1|
|      1088|    1|
|      1645|    1|
|      1507|    1|
|      1084|    1|
|      1395|    1|
|      1721|    1|
|      1460|    1|
|      1025|    1|
|      1127|    1|
|      1483|    1|
|      1896|    1|
|      1522|    1|
|      1990|    1|
|      1352|    1|
+----------+-----+
only showing top 20 rows


In [11]:
print("Insight 2: Average Salary per Gender")
INSIGHT2 = DFCleaned.groupBy("Gender").agg(avg("Annual_Income").alias("Average_Salary"))
INSIGHT2.show()

Insight 2: Average Salary per Gender
+------+------------------+
|Gender|    Average_Salary|
+------+------------------+
|     F| 57.87592954990217|
|     M|56.730997876857764|
+------+------------------+

